# Full bank · generate the answers (math and code)

- **Section 1 (open models): A100.** About 2 hours for both domains.
- **Section 2 (GPT-4.1-mini by API): no GPU.** Run it in a separate CPU runtime (`RUN_OPEN_MODELS = False`, `RUN_API = True`) so it uses no A100 time. About 80 minutes and $6; it stops at $8. Needs the Colab secret `OPENROUTER_API_KEY`. Never run it in two places at once.

The question files (`bank/full/questions_math.jsonl` and `questions_code.jsonl`, from `bank.select_full`) must be in `main`. Math and code can be run together or one at a time (`DOMAINS`). Every finished chunk (200 problems; 50 for the API) is copied to Drive, so if the runtime stops, run the notebook again and it continues. No label is read or made here. Share the Drive folder with the label keeper, who runs notebook 11.

In [ ]:
DOMAINS = ["math", "code"]      # or just one of them
RUN_OPEN_MODELS = True          # section 1: needs an A100
RUN_API = False                 # section 2: needs OPENROUTER_API_KEY, no GPU
DISCONNECT = True               # release the runtime at the end
DRIVE = "/content/drive/MyDrive/cs544_full_bank"
BRANCH = "main"
DOMAINS_ARG = " ".join(DOMAINS)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /content
!rm -rf /content/project && git clone -q -b {BRANCH} https://github.com/Manas-33/CSCI-544-Course-Project.git /content/project
%cd /content/project
!git log --oneline -1
!mkdir -p {DRIVE}

import datetime, os, subprocess
for d in DOMAINS:
    assert os.path.isfile(f"bank/full/questions_{d}.jsonl"), f"bank/full/questions_{d}.jsonl is missing: run bank.select_full and merge its files first"
commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
now = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
with open(f"{DRIVE}/runs.txt", "a") as f:  # the commit is also in every generations_manifest.json
    f.write(f"{now} notebook 10, commit {commit}, domains {DOMAINS}, open models {RUN_OPEN_MODELS}, API {RUN_API}\n")

## 1. Open models (A100)
Phi-4-mini, Granite-3.3-8B and OLMo-2-7B with vLLM, at the pilot's commits and library versions. Each model is loaded once for both domains and is saved to Drive before the next one starts.

In [ ]:
if RUN_OPEN_MODELS:
    !pip install -q vllm==0.32.0 transformers==5.19.0
    for g in ("phi4mini", "granite33", "olmo2"):
        !python -m bank.generate_full run --generator {g} --domains {DOMAINS_ARG} --save {DRIVE}

## 2. GPT-4.1-mini through OpenRouter (no GPU)
The spend is in `api_spend.json` in the Drive folder, a new ledger that does not include the pilot's $1.19. If some answers fail, run the cell again: it asks only for the missing ones.

In [ ]:
if RUN_API:
    import os
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    !python -m bank.generate_full run --generator gpt41mini --domains {DOMAINS_ARG} --save {DRIVE} --budget-usd 8

## 3. What is done, and disconnect

In [ ]:
!python -m bank.generate_full status --domains {DOMAINS_ARG} --gen {DRIVE}/gen
!head -n 2 {DRIVE}/api_spend.json

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
if DISCONNECT:
    runtime.unassign()